# Calculate global mortality with parametric bootstrapping

This may require large memory ~60GB.

In [ ]:
import os
import xarray as xr
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import mortality
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [ ]:
# === Path config ===
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
BMR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR")
TMREL_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "TMREL")
RR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "rr_pm25")

In [ ]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)
pop = pop.reindex_like(masks, method="nearest", tolerance=1e-9)

In [ ]:
# === Calculate the scalar distributions ===
n_samples = 300

# TMREL from GBD21 (uniform distribution)
tmrel_file = f"TMREL_{n_samples}_samples_pm25.nc"
tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
tmrel_da = xr.open_dataarray(tmrel_path)

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]
dates = f"{years.start}-{years.stop}"

PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "annual_pm25_bc")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25" / "global" / f"{n_samples}_samples")

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")

    # RR from GBD21 (normal distribution)
    rr_file = f"RR_{health_VAR}_{n_samples}_samples_pm25.nc"
    rr_path = os.path.join(RR_DIR, rr_file)
    rr_da = xr.open_dataarray(rr_path)

    del rr_file, rr_path

    # Load BMR for each grid point (normal distribution)
    bmr_file = f"GBD_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)

    del bmr_file, bmr_path

    for ens_num in ensemble_members:
        print(f"Processing ensemble member {ens_num:02d}")

        # Load ozone data
        pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM25_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path).astype("float32")

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-9 km distance
        pm25 = pm25.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

        del pm25_file, pm25_path

        # make pm25 dask-backed
        pm25 = pm25.chunk({'lat': 180, 'lon': 360})

        for year in range(years.start, years.stop + 1):
            print(f"Processing year {year}")

            # Find the RR at each grid point
            pm25_year = pm25.sel(year=year)
            RR = rr_da.interp(exposure=pm25_year)

            del pm25_year

            # Calculate the attributable fraction
            AF = (1 - (1/RR)).chunk({"samples": 10})

            del RR

            POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})

            # Calculate mortality at each grid point for n samples
            M = mortality(AF, BMR, POP)

            del AF, POP

            # Calculate the total global mortality
            global_M = M.sum(dim=("lat", "lon"))

            del M

            description = (f"Global {health_VAR} mortality due to PM2.5 "
                           "- scripts by A.F. Wells (2025)")
            global_M.attrs["description"] = description
            global_M.attrs["health_var"] = health_VAR
            global_M.attrs["model"] = model
            global_M.attrs["scenario"] = scenario
            global_M.attrs["ensemble_number"] = ens_num
            global_M.attrs["year"] = year

            out_file = f"Global_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            global_M.to_netcdf(out_path)

            del global_M

        del pm25

    del rr_da, BMR

print("All processing complete.")